# IN-Abs Legal PDF Preprocessor (Steps 0-2)
Run cells top to bottom.

- **Input**: `MyDrive/<DATA_ROOT>/{train,validate,test}/` containing judgment PDFs and summary PDFs
- **Output** (in `MyDrive/inabs_project/`):
  - `processed/<split>/<doc_id>.json`: cleaned, sentence-segmented judgment (what later stages read)
  - `summaries/<split>/<doc_id>.txt`: cleaned reference summary (text only; used for oracle labels and evaluation, never as model input)
  - `match_report.json`, `errors.jsonl`, `manifest.csv`, `pilot_ids.json`
- Resumable: a document is skipped if its output JSON already exists. If Colab disconnects, re-run the same cell.

In [ ]:
# CELL 1: install
!pip install pdfplumber pysbd tqdm --quiet
print('done')

In [ ]:
# CELL 2: mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 3: CONFIG  <-- edit DATA_ROOT to match your Drive folder
import os
DATA_ROOT    = '/content/drive/MyDrive/IN-Abs'           # folder that contains train/ validate/ test/
OUTPUT_DIR   = '/content/drive/MyDrive/inabs_project'
NUM_WORKERS  = 2                                         # free Colab = 2, Colab Pro = 4
SUMMARY_HINT = 'summar'   # a PDF is treated as a SUMMARY if any parent folder name contains this text
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Show the folder structure so you can confirm the hint matches your layout
for sp in ['train', 'validate', 'test']:
    p = f'{DATA_ROOT}/{sp}'
    if not os.path.isdir(p):
        print(f'!! missing folder: {p}'); continue
    print(f'\n{sp}/')
    for d in sorted(os.listdir(p))[:10]:
        full = os.path.join(p, d)
        if os.path.isdir(full):
            n = len([f for f in os.listdir(full) if f.lower().endswith('.pdf')])
            print(f'   {d}/  ({n} pdfs)')
        else:
            print(f'   {d}')

In [ ]:
%%writefile /content/preprocess_inabs.py
"""
IN-Abs preprocessor: PDF judgments -> sentence JSON, PDF summaries -> clean .txt
Resumable (skips docs whose output already exists), split-aware, optional pilot subset.
"""
import os, re, sys, json, time, random, logging, argparse, traceback, unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

try:
    from tqdm import tqdm
except ImportError:
    def tqdm(it=None, total=None, **kw): return it

try:
    import pdfplumber
except ImportError:
    print('ERROR: pip install pdfplumber'); sys.exit(1)

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('preprocess')

SPLITS         = ['train', 'validate', 'test']
MIN_TEXT_CHARS = 200     # judgment: below this => probably scanned (needs OCR)
MIN_WORDS_DOC  = 50
MIN_WORDS_SENT = 5
MIN_SUM_CHARS  = 50      # summaries are shorter than judgments

# ---------------------------------------------------------------- patterns
SECTION_PATTERNS = {
    'FACTS': re.compile(r'^\s*(facts?|background|factual\s+background|brief\s+facts?|statement\s+of\s+facts?|facts\s+of\s+the\s+case)\s*[:.\-]?\s*$', re.I),
    'ISSUES': re.compile(r'^\s*(issues?(\s+(raised|framed|for\s+(determination|consideration)))?|questions?\s+of\s+law|points?\s+(of|for)\s+(determination|consideration))\s*[:.\-]?\s*$', re.I),
    'ARGUMENTS': re.compile(r'^\s*(arguments?|submissions?|contentions?|submissions?\s+(of\s+)?(counsel|parties|appellants?|respondents?|petitioners?))\s*[:.\-]?\s*$', re.I),
    'REASONING': re.compile(r'^\s*(reasoning|analysis|discussion|findings?|ratio(\s+decidendi)?|observations?|our\s+(view|analysis|opinion))\s*[:.\-]?\s*$', re.I),
    'JUDGMENT': re.compile(r'^\s*(judge?ment|order|decree|decision|conclusion|result|disposal|operative\s+(part|portion))\s*[:.\-]?\s*$', re.I),
    'STATUTES': re.compile(r'^\s*(statutes?|(relevant\s+)?(statutory\s+)?provisions?)\s*[:.\-]?\s*$', re.I),
}

BOILERPLATE = [
    re.compile(r'indian\s+kanoon', re.I),
    re.compile(r'take\s+notes\s*\|?\s*print', re.I),
    re.compile(r'try\s+out\s+our\s+premium', re.I),
    re.compile(r'^\s*page\s+\d+\s*(of\s+\d+)?\s*$', re.I),
    re.compile(r'^\s*[-_=*\u2022\s]{3,}$'),
    re.compile(r'^\s*\d{1,4}\s*$'),                 # bare page numbers
]

HYPHEN_RE   = re.compile(r'(\w)-\n(\w)')
NUMPARA_RE  = re.compile(r'^\s*\(?\d{1,3}[.)]\s+\S')   # "12. The appellant..."
ABBREV_RE   = re.compile(r'\b(Mr|Mrs|Ms|Dr|Prof|Vs|vs|No|Nos|Art|Sec|Cl|Sch|J|JJ|CJ|CJI|HC|SC|'
                         r'Co|Ltd|Pvt|Smt|Shri|Sri|Hon|Rs|St|Mt|Dt|Regd|Govt|Dept|Corp|Ors|Anr|'
                         r'i\.e|e\.g|etc|viz|ibid|cf|p|pp|ed|vol|Vol)\.', re.I)

_SEG = None
def get_segmenter():
    """pysbd (handles legal abbreviations better); fallback = regex splitter."""
    global _SEG
    if _SEG is None:
        try:
            import pysbd
            _SEG = ('pysbd', pysbd.Segmenter(language='en', clean=False))
        except Exception:
            _SEG = ('regex', None)
    return _SEG

def split_sentences(text):
    kind, seg = get_segmenter()
    if kind == 'pysbd':
        try:
            return [s.strip() for s in seg.segment(text)]
        except Exception:
            pass
    protected = ABBREV_RE.sub(lambda m: m.group(0).replace('.', '\x00'), text)
    raw = re.split(r'(?<=[.!?])\s+(?=[A-Z"(\u201c])', protected)
    return [s.replace('\x00', '.').strip() for s in raw]

# ---------------------------------------------------------------- extraction
def extract_pages(path):
    pages = []
    try:
        with pdfplumber.open(path) as pdf:
            for pg in pdf.pages:
                try:
                    t = pg.extract_text() or ''
                except Exception:
                    t = ''
                pages.append(t)
    except Exception:
        return []
    return pages

def strip_repeated_lines(pages):
    """Remove running headers/footers: short lines (digits masked) repeating on many pages."""
    if len(pages) < 4:
        return pages
    norm = lambda l: re.sub(r'\d+', '#', l.strip().lower())
    cnt = Counter()
    for p in pages:
        for l in {norm(x) for x in p.split('\n') if x.strip()}:
            cnt[l] += 1
    thr = max(3, int(0.4 * len(pages)))
    bad = {l for l, c in cnt.items() if c >= thr and len(l) < 100}
    out = ['\n'.join(x for x in p.split('\n') if norm(x) not in bad) for p in pages]
    # safeguard: if stripping removed >30% of the text, it was too aggressive -> keep original
    if sum(len(p) for p in out) < 0.7 * sum(len(p) for p in pages):
        return pages
    return out

def clean_text(raw):
    t = unicodedata.normalize('NFKC', raw)
    t = t.replace('\u2018', "'").replace('\u2019', "'").replace('\u201c', '"').replace('\u201d', '"')
    t = t.replace('\u2013', '-').replace('\u2014', '-')
    t = HYPHEN_RE.sub(r'\1\2', t)
    lines = [l for l in t.split('\n') if not any(p.search(l) for p in BOILERPLATE)]
    t = '\n'.join(re.sub(r'[ \t]{2,}', ' ', l).rstrip() for l in lines)
    return re.sub(r'\n{3,}', '\n\n', t).strip()

def judgment_to_sentences(cleaned):
    """Single ordered pass: sentence order is preserved; each sentence gets the section active when it appeared."""
    sentences, buf, cur = [], [], 'PREAMBLE'
    idx = 0
    def flush():
        nonlocal buf, idx
        if buf:
            para = re.sub(r'\s+', ' ', ' '.join(buf)).strip()
            para = re.sub(r'^\(?\d{1,3}[.)]\s+', '', para)      # drop leading paragraph number "12."
            for s in split_sentences(para):
                if len(s.split()) >= MIN_WORDS_SENT:
                    sentences.append({'id': idx, 'text': s, 'section': cur, 'n_tokens': len(s.split())})
                    idx += 1
        buf = []
    for line in cleaned.split('\n'):
        if not line.strip():
            flush(); continue
        hit = next((lbl for lbl, pat in SECTION_PATTERNS.items() if pat.match(line)), None)
        if hit:
            flush(); cur = hit; continue
        if NUMPARA_RE.match(line):
            flush()
        buf.append(line.strip())
    flush()
    return sentences

# ---------------------------------------------------------------- per-doc worker
def process_one(job):
    split, doc_id, jpath, spath, out_json, out_sum = job
    res = {'doc_id': doc_id, 'split': split, 'status': 'error'}
    try:
        pages = strip_repeated_lines(extract_pages(jpath))
        raw = '\n'.join(pages)
        if len(raw.strip()) < MIN_TEXT_CHARS:
            res.update(status='empty', reason=f'judgment only {len(raw.strip())} chars (scanned? needs OCR)')
            return res
        cl = clean_text(raw)
        if len(cl.split()) < MIN_WORDS_DOC:
            res.update(status='empty', reason=f'judgment only {len(cl.split())} words'); return res
        sents = judgment_to_sentences(cl)
        if not sents:
            res.update(status='empty', reason='no sentences after segmentation'); return res

        sec_counts = Counter(s['section'] for s in sents)
        rec = {
            'doc_id': doc_id, 'split': split, 'filename': os.path.basename(jpath),
            'status': 'ok',
            'stats': {
                'num_pages': len(pages), 'word_count': len(cl.split()), 'char_count': len(cl),
                'num_sentences': len(sents),
                'avg_tokens_per_sentence': round(sum(s['n_tokens'] for s in sents) / len(sents), 2),
                'max_tokens_per_sentence': max(s['n_tokens'] for s in sents),
                'sections_found': list(sec_counts), 'section_counts': dict(sec_counts),
            },
            'sentences': sents,
        }
        # reference summary: text only, stored separately (never used at inference)
        sum_words = 0
        if spath:
            sraw = '\n'.join(extract_pages(spath))
            stxt = re.sub(r'\s+', ' ', clean_text(sraw)).strip()
            if len(stxt) >= MIN_SUM_CHARS:
                os.makedirs(os.path.dirname(out_sum), exist_ok=True)
                with open(out_sum + '.tmp', 'w', encoding='utf-8') as f: f.write(stxt)
                os.replace(out_sum + '.tmp', out_sum)
                sum_words = len(stxt.split())
            else:
                res['summary_warning'] = f'summary only {len(stxt)} chars'
        rec['stats']['summary_words'] = sum_words
        rec['has_summary'] = sum_words > 0

        os.makedirs(os.path.dirname(out_json), exist_ok=True)
        with open(out_json + '.tmp', 'w', encoding='utf-8') as f:
            json.dump(rec, f, ensure_ascii=False)
        os.replace(out_json + '.tmp', out_json)      # atomic: no half-written files on crash
        res.update(status='ok', n_sent=len(sents), sum_words=sum_words)
    except Exception as e:
        res.update(status='error', reason=str(e), trace=traceback.format_exc())
    return res

# ---------------------------------------------------------------- file discovery / matching
def norm_key(p):
    s = Path(p).stem.lower()
    s = re.sub(r'(^|[_\-\s])(summary|summaries|summ|abstract|judg(e)?ment|judgements?)($|[_\-\s])', ' ', s)
    return re.sub(r'[^a-z0-9]+', '', s)

def is_summary_path(p, split_dir, sum_hint):
    rel = [x.lower() for x in Path(p).relative_to(split_dir).parts[:-1]]
    return any(sum_hint in x for x in rel)

def discover(input_dir, split, sum_hint='summar'):
    sdir = Path(input_dir) / split
    if not sdir.exists():
        # tolerate 'val' / 'validation'
        alts = {'validate': ['validation', 'val', 'dev']}.get(split, [])
        for a in alts:
            if (Path(input_dir) / a).exists(): sdir = Path(input_dir) / a; break
    pdfs = sorted(p for p in sdir.rglob('*') if p.suffix.lower() == '.pdf') if sdir.exists() else []
    J, S = {}, {}
    for p in pdfs:
        (S if is_summary_path(p, sdir, sum_hint) else J)[norm_key(p)] = p
    return J, S

def build_jobs(input_dir, output_dir, sum_hint, pilot=None, seed=42, first=False):
    per_split, report = {}, {}
    for sp in SPLITS:
        J, S = discover(input_dir, sp, sum_hint)
        keys = sorted(J)
        report[sp] = {'judgments': len(J), 'summaries': len(S),
                      'judgments_without_summary': sorted(set(J) - set(S))[:20],
                      'n_no_summary': len(set(J) - set(S)),
                      'n_orphan_summaries': len(set(S) - set(J))}
        per_split[sp] = [(k, J[k], S.get(k)) for k in keys]
    pilot_file = os.path.join(output_dir, 'pilot_ids.json')
    if pilot:
        chosen = {}
        for sp, n in zip(SPLITS, pilot):
            withsum = [k for k, j, s in per_split[sp] if s is not None]   # sorted by filename
            if first:
                chosen[sp] = withsum[:n]                                   # first N by filename
            else:
                order = withsum[:]
                random.Random(seed).shuffle(order)                         # same seed => bigger pilots contain smaller ones
                chosen[sp] = sorted(order[:n])
        os.makedirs(output_dir, exist_ok=True)
        json.dump(chosen, open(pilot_file, 'w'), indent=1)
        for sp in SPLITS:
            keep = set(chosen.get(sp, []))
            per_split[sp] = [t for t in per_split[sp] if t[0] in keep]
    jobs = []
    for sp in SPLITS:
        for k, jp, spth in per_split[sp]:
            doc_id = f'{sp}_{k}'[:120]
            out_json = os.path.join(output_dir, 'processed', sp, f'{doc_id}.json')
            out_sum  = os.path.join(output_dir, 'summaries', sp, f'{doc_id}.txt')
            jobs.append((sp, doc_id, str(jp), str(spth) if spth else None, out_json, out_sum))
    return jobs, report

def run(input_dir, output_dir, workers, pilot, sum_hint, seed, first=False):
    os.makedirs(output_dir, exist_ok=True)
    jobs, report = build_jobs(input_dir, output_dir, sum_hint, pilot, seed, first)
    json.dump(report, open(os.path.join(output_dir, 'match_report.json'), 'w'), indent=1)
    for sp, r in report.items():
        log.info(f"  {sp:<9} judgments={r['judgments']:<5} summaries={r['summaries']:<5} "
                 f"no_summary={r['n_no_summary']}  orphan_summaries={r['n_orphan_summaries']}")
    todo = [j for j in jobs if not os.path.exists(j[4])]          # resumable
    log.info(f'  total selected={len(jobs):,}  already done={len(jobs)-len(todo):,}  to process={len(todo):,}')
    if not todo:
        log.info('Nothing to do.'); return
    errs = os.path.join(output_dir, 'errors.jsonl')
    t0, c = time.time(), Counter()
    with ProcessPoolExecutor(max_workers=workers) as pool:
        futs = {pool.submit(process_one, j): j for j in todo}
        bar = tqdm(as_completed(futs), total=len(todo), desc='Preprocessing', unit='doc')
        for f in bar:
            j = futs[f]
            try:
                r = f.result(timeout=300)
            except Exception as e:
                r = {'doc_id': j[1], 'split': j[0], 'status': 'error', 'reason': repr(e)}
            c[r['status']] += 1
            if r['status'] != 'ok' or r.get('summary_warning'):
                with open(errs, 'a', encoding='utf-8') as ef:
                    ef.write(json.dumps({k: v for k, v in r.items() if k != 'trace'}) + '\n')
    log.info(f"  DONE  ok={c['ok']}  empty={c['empty']}  error={c['error']}  time={(time.time()-t0)/60:.1f} min")

if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--input_dir', required=True)
    ap.add_argument('--output_dir', required=True)
    ap.add_argument('--workers', type=int, default=2)
    ap.add_argument('--pilot', type=str, default=None, help='counts as train,validate,test e.g. 100,100,500')
    ap.add_argument('--summary_hint', default='summar', help="substring identifying summary folders")
    ap.add_argument('--seed', type=int, default=42)
    ap.add_argument('--first', action='store_true', help='take the first N by filename instead of a random sample')
    a = ap.parse_args()
    pilot = [int(x) for x in a.pilot.split(',')] if a.pilot else None
    run(a.input_dir, a.output_dir, a.workers, pilot, a.summary_hint, a.seed, a.first)


In [ ]:
# CELL 5: SMOKE TEST on 3/1/1 docs, writes to local disk (not Drive). Check the output, then move on.
!python /content/preprocess_inabs.py --input_dir "{DATA_ROOT}" --output_dir /content/smoke --workers 2 --pilot 3,1,1 --summary_hint "{SUMMARY_HINT}"

import json, glob
print('\n=== match report ===')
print(json.dumps(json.load(open('/content/smoke/match_report.json')), indent=1)[:1500])
files = sorted(glob.glob('/content/smoke/processed/*/*.json'))
print(f'\n{len(files)} JSONs written')
if files:
    d = json.load(open(files[0]))
    print(d['doc_id'], d['stats'])
    for s in d['sentences'][:6]:
        print(f"  [{s['section']:<9}] {s['n_tokens']:3d} tok  {s['text'][:90]}")
    sp = files[0].replace('/processed/', '/summaries/').replace('.json', '.txt')
    print('\nSUMMARY:', open(sp).read()[:300] if os.path.exists(sp) else 'NO SUMMARY FILE')

In [ ]:
# CELL 6: PILOT RUN, saved to Drive.   --pilot is ordered: train,validate,test
# Here: 100 train, 100 validate, 500 test, taking the FIRST N by filename (--first).
# Remove --first to take a random sample instead (seed=42, saved to pilot_ids.json).
!python /content/preprocess_inabs.py --input_dir "{DATA_ROOT}" --output_dir "{OUTPUT_DIR}" --workers {NUM_WORKERS} --pilot 100,100,500 --first --summary_hint "{SUMMARY_HINT}"

In [ ]:
# CELL 7: FULL RUN (all splits). Run only after the pilot QC (Cell 8) looks good.
# Pilot docs are skipped automatically (already done). Safe to re-run after a disconnect.
import os
_p = f'{OUTPUT_DIR}/pilot_ids.json'
if os.path.exists(_p):
    os.rename(_p, f'{OUTPUT_DIR}/pilot_ids_used.json')   # keep a record, but disable the pilot filter
!python /content/preprocess_inabs.py --input_dir "{DATA_ROOT}" --output_dir "{OUTPUT_DIR}" --workers {NUM_WORKERS} --summary_hint "{SUMMARY_HINT}"

In [ ]:
# CELL 8: QC REPORT + manifest.csv  (run after the pilot AND after the full run)
import os, json, glob, statistics as st, csv
rows = []
for sp in ['train', 'validate', 'test']:
    for f in glob.glob(f'{OUTPUT_DIR}/processed/{sp}/*.json'):
        d = json.load(open(f)); s = d['stats']
        rows.append(dict(doc_id=d['doc_id'], split=sp, sentences=s['num_sentences'], words=s['word_count'],
                         summary_words=s['summary_words'], has_summary=d['has_summary'],
                         sections=len(s['sections_found'])))
with open(f'{OUTPUT_DIR}/manifest.csv', 'w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)

print(f'{"split":<9}{"docs":>6}{"no_summ":>9}{"med_sents":>11}{"med_words":>11}{"med_sum_w":>11}{"ratio":>8}')
for sp in ['train', 'validate', 'test']:
    r = [x for x in rows if x['split'] == sp]
    if not r: continue
    sm = [x['summary_words'] for x in r if x['has_summary']]
    ratio = st.median([x['summary_words']/x['words'] for x in r if x['has_summary'] and x['words']]) if sm else 0
    print(f'{sp:<9}{len(r):>6}{sum(not x["has_summary"] for x in r):>9}'
          f'{st.median(x["sentences"] for x in r):>11.0f}{st.median(x["words"] for x in r):>11.0f}'
          f'{(st.median(sm) if sm else 0):>11.0f}{ratio:>8.2f}')

errs = f'{OUTPUT_DIR}/errors.jsonl'
if os.path.exists(errs):
    e = [json.loads(l) for l in open(errs)]
    print(f'\nProblem docs: {len(e)}  (empty = likely scanned, needs OCR)')
    for x in e[:10]: print('  ', x['doc_id'], '|', x.get('reason') or x.get('summary_warning'))
else:
    print('\nNo problem docs logged.')

print('\nCHECKS:')
print(' - every doc should have has_summary=True (no_summ column ~0)')
print(' - extraction failures (errors.jsonl) should be under ~5%')
print(' - med_sum_w is your summary length budget for Step 9 (MMR selection)')

In [ ]:
# CELL 9: spot-check a random document (read the sentences yourself!)
import random, glob, json, os
f = random.choice(glob.glob(f'{OUTPUT_DIR}/processed/*/*.json'))
d = json.load(open(f))
print(d['doc_id'], d['stats'], '\n')
for s in d['sentences'][:12]:
    print(f"[{s['section']:<9}] {s['n_tokens']:3d}  {s['text'][:110]}")
print('\n--- reference summary ---')
sp = f.replace('/processed/', '/summaries/').replace('.json', '.txt')
print(open(sp).read()[:600] if os.path.exists(sp) else 'none')